## Iteration 3 of History Matching for a stochastic SIR model

In this example, we assume that prevalence has been observed at two distinct times.  For iteration 2, we will "cut" down parameter space using only the observation of prevalence at the FIRST time point.

In [1]:
# First load some libraries, including History Matching
%load_ext autoreload
%autoreload 2

import os
import re
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyDOE import lhs
from history_matching import HistoryMatching, HistoryMatchingCut, quick_read, Basis

sys.path.append("..") # Adds higher directory to python modules path.
from sir import SIR

Looks like you don't have CUDA, that's okay, we'll try using CPU but it will be SLOW!


### Configure parameters and data for History Matching
Note: This is (and must be) the same as on iteration 0.  Ordinarily I would read from file.

In [2]:
iteration = int(re.search(r'iter(\d+)', os.getcwd()).group(1)) # Index of the current iteration
n_samples_per_iter = 500 # Number of simulations to conduct on this iteration

In [3]:
# The implausibility threshold determines how willing we are to retain regions
# of parameter space that are inconsistent with the underlying data. A higher
# threshold is more risk averse in that potentially good regions are less likely
# to be rejected, however it will take more iterations/simulations to achieve results.
implausibility_threshold = 3
training_fraction = 0.75 # Fraction of simulations to use as training
discrepancy_std = 3 # Accounts for uncertainty w.r.t model structure

In [4]:
# Observed data
observations = pd.DataFrame({
    'Times': [3, 15],
    'Prevalence': [15, 40],
    'Stdev': [4, 2.3],
})
print(observations)

# For this first iteration, we're going to make one "cut" using the first observation, but 
# you can separately do multiple "cuts" per iteration using several obervations (separately)
# We'll need a name for this cut and the desired results
cut_name = 'Prevalence_Meas' # No spaces or strange characters!

desired_result_idx = 1 # Second time point
desired_result = observations.iloc[desired_result_idx]['Prevalence']
desired_result_var = observations.iloc[desired_result_idx]['Stdev']**2

   Times  Prevalence  Stdev
0      3          15    4.0
1     15          40    2.3


In [5]:
# Here we define the parameter names and ranges
param_info = pd.DataFrame({
    'Name':['Beta', 'Gamma'],
    'Min':[1e-6, 1e-6],
    'Max':[0.01, 0.5]
}).set_index('Name')
params = param_info.index.values
n_params = param_info.shape[0] # We'll use this one place later
print(param_info)

            Min   Max
Name                 
Beta   0.000001  0.01
Gamma  0.000001  0.50


### Load samples selected on the previous iteration and run simulations for this iteration. Also process and plot results

In [6]:
# For the first iteration, the samples are random.  We'll use Latin Hypercube Sampling
# to make the samples more uniformly random.
# samples should be a pandas data frame, and must have an index named 'Sample_Id'
samples = pd.read_csv(os.path.join('..', 'iter%d'%(iteration-1), 'Candidates_for_iter%d.csv'%iteration)).loc[:n_samples_per_iter]
samples.index.name = 'Sample_Id'

# Plot the samples
f, ax = plt.subplots(figsize=(6,6))
ax.scatter(x=samples[param_info.index.values[0]], y=samples[param_info.index.values[1]])
ax.set_xlim([param_info['Min'][0], param_info['Max'][0]])
ax.set_ylim([param_info['Min'][1], param_info['Max'][1]])
ax.set_xlabel(param_info.index.values[0])
ax.set_ylabel(param_info.index.values[1])

FileNotFoundError: [Errno 2] File b'../iter2/Candidates_for_iter3.csv' does not exist: b'../iter2/Candidates_for_iter3.csv'

In [ ]:
# Run the simulations specified by the samples and plot the results

f = plt.figure(figsize=(16,10))
sim_results = []
for idx, sample in samples.iterrows():
    z = SIR(beta=sample['Beta'], gamma=sample['Gamma'])
    T,_,P = z.sim() # Run the simulation
    prevalence = [p[1] for p in P] # Analyze the simulation to get the prevalence
    
    # Because we used SSA, the time vector T does not contain the prevalence at the 
    # exact observation time, t_obs.  Let's find the first measurement after t_obs
    # Here I'm getting both observations, but we really only need one
    for i, t_obs in enumerate(observations['Times']):
        value = next((p[1] for t,p in zip(T,P) if t>t_obs), None)
        if not value:
            value = P[-1][1]
        sim_results.append([idx, i, t_obs, value])
    
    # Plot
    plt.plot(T,prevalence)

# Convert sim_results into a pandas DataFrame
sim_results = pd.DataFrame(sim_results, columns=['Sample_Id', 'ObsIdx', 'ObsTime', 'Prevalence'])

# Simulation results will ultimately need a 'Sample_Id' and 'Sim_Id'
# The Sample_Id corresponds to the index in the samples dataframe above
# You can do more than one replicate of each sample, in which case you'd
# end up wiht more than one Sim_Id per Sample_Id.  We're not doing that here,
# so we can basically set Sim_Id to anything.
sim_results['Sim_Id'] = sim_results['Sample_Id']
    
# Plot the observations over the realized trajectories
for i,obs in observations.iterrows():
    plt.plot(obs['Times'], obs['Prevalence'], 'ko')
    plt.plot(
        [obs['Times'],obs['Times']], 
        [obs['Prevalence']-2*obs['Stdev'],obs['Prevalence']+2*obs['Stdev']],
        'k-')

In [ ]:
# sim_results contains the simulated values at both observation times, but for this iteration
# we only want to use the SECOND one (ObsIdx == 1).
# Also note that results must be a Series with index containing 'Sample_Id' and 'Sim_Id'
results = sim_results \
    .query('ObsIdx==@desired_result_idx')[['Sample_Id', 'Sim_Id', 'Prevalence']] \
    .set_index(['Sample_Id', 'Sim_Id'])['Prevalence']
print(results.tail())

## Now begins History Matching!
### First step is to emulate the selected simulation output

In [ ]:
# Finally we get to do some History Matching!

# Begin by creating an instance of the HistoryMatching class
hm = HistoryMatching(
    cut_name = cut_name,
    param_info = param_info,
    inputs = samples,
    results = results,
    desired_result = desired_result,
    desired_result_var = desired_result_var,
    iteration = iteration,
    implausibility_threshold = implausibility_threshold,
    discrepancy_var = discrepancy_std**2,
    training_fraction = training_fraction
)
hm.save() # Save to disk


In [ ]:
# Now we begin the process of emulating the simulation output
# This process contains two steps.  The first step is to fit a deterministic model, here
# we use a generalized linear model (GLM).  The glm will attempt to model the output (prevalence 
# at the first observation) as a function of some inputs.  Those inputs need not be the model
# parameters directly!  The inputs could be anything from a constant intercept up to third or higher
# order interaction terms between parameters.  The following Basis instance builds out the GLM input
# parameters from the overall simulation input parameters.
#
# Some strategy is required when choosing these.  If you know which parameters matter, there's a way
# to directly specify those parameters.  Alternatively, if you have no idea, you can initially include 
# an intercept, first, second, and maybe also third order interaction terms.  The Basis class has a 
# built-in penalized regression that throws away unneeded terms (basis vectors).
# The second step of emulation, as demonstrated here, fits a GPR to the redisual error between the 
# simulated outputs and the GLM estimates.  If the GLM fits really well, the residual is mostly noise and
# the GPR has a hard time fitting / isn't very informative.  I actually prefer to weaken the GLM enough
# to leave plenty of residual signal for the GPR.  Here, I use only first-order (beta and gamma) terms.
basis_glm = Basis.polynomial_basis(
    params = param_info.index.values,
    intercept = True,
    first_order = True,
    second_order = False,
    third_order = False,
    param_info = param_info)

In [ ]:
# Now fit the glm and plot

### GLM ###############################################################
print("="*80, "\nGeneralized Linear Modeling\n", "="*80)
#######################################################################
f = hm.glm(
    basis = basis_glm,
    family = 'Gaussian',
    force_optimize_glm = True,
    glm_fit_maxiter = 100000,
    plot = True, #force_optimize_glm,
    plot_data = True
)

# Results get saved to disk, so load and display:
from wand.image import Image as WImage
import glob, os
from IPython.display import display
for file in glob.glob(os.path.join(hm.glmdir, "*.pdf")):
    img = WImage(filename=file)
    print(file)
    display(img)

In [ ]:
basis_gpr = Basis.polynomial_basis(
    params=param_info.index.values, 
    intercept = False, 
    first_order=True, 
    param_info=param_info)

In [ ]:
### GPR ###############################################################
print("="*80, "\nGaussian Process Regression\n", "="*80)
#######################################################################
hm.gpr(
    basis = basis_gpr,
    force_optimize_gpr = True,

    sigma2_f_guess = 0.6,
    sigma2_f_bounds = (0.1, 1000),
    sigma2_n_guess =  2.0,
    sigma2_n_bounds = (0.01, 100),

    #lengthscale_guess = [0.09844299, 0.1256657, 0.0976875, 0.09889085, 0.1051974, 0.0950809, 0.10032171, 0.10599185, 0.10627393, 0.09950996, 0.09445544, 0.10285915, 0.10007409, 0.09847433, 0.08963389, 0.10205652, 0.09360044, 0.1024141, 0.09786228, 0.10247492, 0.09852253, 0.09632744, 0.09997534, 0.10767302, 0.10095249, 0.09941825, 0.10214923, 0.10221497, 0.09734157, 0.09093285, 0.10780673, 0.09881377, 0.10597152],
    lengthscale_guess = 0.25,
    lengthscale_bounds = (0.01, 100),

    optimize_sigma2_n = True,
    log_transform = False,

    verbose = True,
    optimizer_options = {
        'eps': 5e-3,
        'disp': True,
        'maxiter': 15000,
        'ftol': 2 * np.finfo(float).eps,
        'gtol': 2 * np.finfo(float).eps,
    },
    plot = True, #force_optimize_gpr,
    plot_data = False
)


In [ ]:
# Results get saved to disk, so load and display:
from wand.image import Image as WImage
import glob, os
from IPython.display import display
for file in glob.glob(os.path.join(hm.gprdir, "*.pdf")):
    img = WImage(filename=file)
    print(file)
    display(img)

### Now use the emulator to calculate "implausibility."  New simulations will not be commissioned on future iterations where implasibility is high.

In [ ]:
### Implausibility ############################################################
print("="*80, "\nImplausibility\n", "="*80)
###############################################################################
hm.calc_and_plot_implausibility(
    plot = True,
    do_plot_data = True,
    plot_data_highlight = pd.DataFrame() #hm.test_data.loc['prime.000049']
) 
    #plot_data_highlight=pd.DataFrame() # plot_data_highlight=hm.training_data.loc['prime.000049']

hm.training_data.to_excel(os.path.join('Cuts', cut_name, 'train_data.xlsx'))
hm.test_data.to_excel(os.path.join('Cuts', cut_name, 'test_data.xlsx'))

print('Good')


In [ ]:
# Results get saved to disk, so load and display:
from wand.image import Image as WImage
import glob, os
from IPython.display import display
for file in glob.glob(os.path.join(hm.combineddir, "*.pdf")):
    img = WImage(filename=file)
    print(file)
    display(img)

# Towards rajectory selection
At this point, we're not making any more progess - we're done!
However, the simulated trajectories still don't match the data very well.
This is because ths model is very noisy.  What we will do is to run may simulations from the non-implausible region of parameter space, and then pick the best ones.  You could get fancy here.

In [ ]:
# Let's throw many darts, then keep the best ones
n_final_samples = 10000

### Cut #######################################################################
print("="*80, "\nCut\n", "="*80)
###############################################################################
# History Matching!
hmc = HistoryMatchingCut(
    cut_dir = 'Cuts',
    iteration = int(re.search(r'iter(\d+)', os.getcwd()).group(1))
)

(_, rejected_percent) = hmc.cut(num_desired_candidates=n_final_samples, constraint = None)

In [ ]:
# Samples for the next iteration are saved to file, in this case it's "Candidates_for_iter1.csv"
# Just to see what they look like, we'll read that file and plot the samples
# Notice how the entire bottom right of the paramter space is empty?  Those points were rejected!

samples = pd.read_csv('Candidates_for_iter%d.csv'%(iteration+1))

f, ax = plt.subplots(figsize=(6,6))
ax.scatter(x=samples[param_info.index.values[0]], y=samples[param_info.index.values[1]])
ax.set_xlim([param_info['Min'][0], param_info['Max'][0]])
ax.set_ylim([param_info['Min'][1], param_info['Max'][1]])
ax.set_xlabel(param_info.index.values[0])
ax.set_ylabel(param_info.index.values[1]);

# Now look at Trajectory Selection ipynb in this directory.